Read Data

In [60]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST','localhost')}:"
    f"{os.getenv('DB_PORT','5433')}/"
    f"{os.getenv('DB_NAME')}"
)

engine = create_engine(DATABASE_URL)

df = pd.read_sql("""
SELECT *
FROM study_events
""", engine)

df.head()

,id,user_id,topic_tag,leetcode_id,difficulty,minutes_spent,outcome,ts
0,1,1,trees,140,medium,55,solved_after_hint,2026-06-03 22:43:51.540590+00:00
1,2,1,dp,498,medium,71,solved_after_hint,2026-06-02 22:43:51.540590+00:00
2,3,1,ml_basics,85,easy,52,solved,2026-06-13 22:43:51.540590+00:00
3,4,1,sql,247,hard,56,failed,2026-06-12 22:43:51.540590+00:00
4,5,1,dp,23,medium,72,solved,2026-06-10 22:43:51.540590+00:00


Check balance of labels

In [61]:
labels = pd.read_sql(""" 
SELECT * FROM labels""", engine)
labels["next_success_7d"].value_counts(normalize=True)
labels.isnull().sum()


id                 0
user_id            0
next_success_7d    0
created_at         0
dtype: int64

In [62]:
data = pd.read_sql("""
    SELECT 
        u.id AS user_id,
        u.email,
        u.display_name,
        se.id AS event_id,
        se.topic_tag,
        se.leetcode_id,
        se.difficulty,
        se.minutes_spent,
        se.outcome,
        se.ts AS event_timestamp,
        l.next_success_7d
    FROM users AS u
    LEFT JOIN study_events AS se ON u.id = se.user_id
    LEFT JOIN labels AS l ON u.id = l.user_id
""", engine)
data.head()
print(data.isnull().sum())
print(data.shape)
data.info()
data["next_success_7d"].isnull().sum()

user_id            0
email              0
display_name       0
event_id           0
topic_tag          0
leetcode_id        0
difficulty         0
minutes_spent      0
outcome            0
event_timestamp    0
next_success_7d    0
dtype: int64
(14972, 11)
<class 'pandas.DataFrame'>
RangeIndex: 14972 entries, 0 to 14971
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   user_id          14972 non-null  int64              
 1   email            14972 non-null  str                
 2   display_name     14972 non-null  str                
 3   event_id         14972 non-null  int64              
 4   topic_tag        14972 non-null  str                
 5   leetcode_id      14972 non-null  int64              
 6   difficulty       14972 non-null  str                
 7   minutes_spent    14972 non-null  int64              
 8   outcome          14972 non-null  str                
 9   eve

np.int64(0)

In [ ]:
data["event_timestamp"] = pd.to_datetime(data["event_timestamp"])
data = data.sort_values("event_timestamp")
# print(data.loc[df['user_id'] == 1])
# Get only rows for User 1, and only show their topic and minutes spent
train_idx = int  (len(data) * .7)
val_end_idx = int(len(data) * .85)
print(data.size)
train = data.iloc[:train_idx]
val = data.iloc[train_idx:val_end_idx]
test = data.iloc[val_end_idx:]

print(train)

164692
       user_id                email display_name  event_id      topic_tag  \
1574        22   user21@example.com      User 21      1575             dp   
5179        71   user70@example.com      User 70      5180             dp   
11844      159  user158@example.com     User 158     11845   two_pointers   
473          7    user6@example.com       User 6       474      ml_basics   
11845      159  user158@example.com     User 158     11846      ml_basics   
...        ...                  ...          ...       ...            ...   
14082      189  user188@example.com     User 188     14083  system_design   
8506       113  user112@example.com     User 112      8507      ml_basics   
12421      167  user166@example.com     User 166     12422  system_design   
601          8    user7@example.com       User 7       602             dp   
13281      178  user177@example.com     User 177     13282      ml_basics   

       leetcode_id difficulty  minutes_spent            outcome  \
1